# 🌸 SweetGF - One-Click Colab Setup

This notebook provides a simplified setup for SweetGF on Google Colab.

## Quick Start:
1. **Change runtime to GPU**: Runtime → Change runtime type → GPU (T4)
2. **Run the cells below** in order
3. **Follow the prompts** for configuration
4. **Access your app** via the provided URL

---

## Step 1: Upload Your Files

Choose one method:

### Method A: Clone from GitHub (Recommended)

In [ ]:
# Replace with your GitHub repository URL
GITHUB_REPO = "https://github.com/yourusername/SweetGF.git"

!git clone {GITHUB_REPO} /content/SweetGF
%cd /content/SweetGF

### Method B: Upload Files Manually

Run this cell, then upload your files using the file browser on the left.

In [ ]:
# Create directory and wait for manual upload
!mkdir -p /content/SweetGF
%cd /content/SweetGF

print("📁 Please upload these files to /content/SweetGF:")
print("   - app.py")
print("   - requirements.txt")
print("   - static/ folder")
print("   - .env.example (optional)")
print("\n⏸️  Pause here and upload files, then continue to next cell.")

---

## Step 2: Run Automated Setup

This will:
- Install all dependencies
- Set up Ollama and download AI model
- Optionally install ComfyUI
- Configure and start the app
- Create a public URL

In [ ]:
%%writefile /content/SweetGF/setup_colab.py
import os
import sys
import subprocess
import time

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("🔧 Installing dependencies...")
run("apt-get update -qq && apt-get install -y -qq wget git")
run("pip install -q flask flask-cors requests python-dotenv pyngrok")

print("\n🤖 Installing Ollama...")
if not os.path.exists("/usr/local/bin/ollama"):
    run("curl -fsSL https://ollama.com/install.sh | sh")

print("\n🚀 Starting Ollama...")
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

print("\n📥 Choose AI model:")
print("1. phi3:mini (2.3GB) - Fast")
print("2. mistral:7b (4.1GB) - Better quality")
choice = input("Enter choice [1]: ").strip() or "1"
model = "phi3:mini" if choice == "1" else "mistral:7b"

print(f"\n📦 Downloading {model}...")
run(f"ollama pull {model}")

print("\n🎨 Install ComfyUI? (for image generation)")
install_comfy = input("y/n [n]: ").strip().lower() == 'y'

if install_comfy:
    print("\n🎨 Setting up ComfyUI...")
    if not os.path.exists('/content/ComfyUI'):
        run("git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI")
        run("cd /content/ComfyUI && pip install -q -r requirements.txt")
    
    os.makedirs('/content/ComfyUI/models/checkpoints', exist_ok=True)
    subprocess.Popen(
        ['python', '/content/ComfyUI/main.py', '--listen', '0.0.0.0', '--port', '8188'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    time.sleep(10)

print("\n⚙️ Configuring environment...")
env = f"""OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_MODEL={model}
COMFYUI_BASE_URL=http://127.0.0.1:8188
COMFYUI_PATH=/content/ComfyUI
FLASK_PORT=5000
FLASK_DEBUG=False
ENABLE_FACE_DETAILER=false
ENABLE_LORA=false
"""
with open('/content/SweetGF/.env', 'w') as f:
    f.write(env)

print("\n✅ Setup complete!")
print(f"Model: {model}")
print(f"ComfyUI: {'Installed' if install_comfy else 'Skipped'}")

In [ ]:
# Run the setup
%cd /content/SweetGF
!python setup_colab.py

---

## Step 3: Setup Public Access (Ngrok)

Get your free ngrok token: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok, conf
import getpass

# Enter your ngrok auth token
NGROK_TOKEN = getpass.getpass("Enter ngrok auth token: ")
conf.get_default().auth_token = NGROK_TOKEN

print("✅ Ngrok configured!")

---

## Step 4: Start SweetGF

This will start the app and create a public URL.

In [ ]:
import threading
import time
from pyngrok import ngrok

%cd /content/SweetGF

# Start Flask in background
def run_flask():
    import os
    os.system('python app.py')

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

print("🚀 Starting Flask app...")
time.sleep(5)

# Create public URL
public_url = ngrok.connect(5000, bind_tls=True)

print("\n" + "="*60)
print("🎉 SweetGF is now running!")
print("="*60)
print(f"\n🌐 Access your app: {public_url}")
print("\n💡 Tips:")
print("   - Share the URL with anyone")
print("   - First message may take longer")
print("   - Keep this notebook running")
print("\n" + "="*60)

---

## Step 5: Monitor Status (Optional)

In [ ]:
import requests

def check(url, name):
    try:
        r = requests.get(url, timeout=5)
        print(f"✅ {name}: Running ({r.status_code})")
    except Exception as e:
        print(f"❌ {name}: {str(e)}")

print("Checking services...\n")
check("http://localhost:11434", "Ollama")
check("http://localhost:8188", "ComfyUI")
check("http://localhost:5000/api/health", "SweetGF")

---

## Step 6: Keep Alive

Run this to prevent Colab from disconnecting.

In [ ]:
import time

print("💤 Keeping session alive...")
print("Your app is running at the URL above.")
print("Press ⏹️ Stop button to end.\n")

try:
    while True:
        time.sleep(60)
        print(".", end="", flush=True)
except KeyboardInterrupt:
    print("\n👋 Stopping...")

---

## Troubleshooting

### Restart Ollama

In [ ]:
!pkill ollama
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)
print("✅ Ollama restarted")

### Restart ComfyUI

In [ ]:
!pkill -f "python main.py"
import subprocess, time
subprocess.Popen(
    ['python', '/content/ComfyUI/main.py', '--listen', '0.0.0.0', '--port', '8188'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(10)
print("✅ ComfyUI restarted")

### Check GPU

In [ ]:
!nvidia-smi

### View Logs

In [ ]:
# Check if files exist
!ls -lh /content/SweetGF/

# Check Ollama models
!ollama list